# Анализ коэффициентов пролонгации договоров

- Автор: Романов Артём
- Дата: 17.04.2026

## Цели и задачи проекта

Необходимо оценить, насколько эффективно аккаунт-менеджеры отдела сопровождения клиентов справляются с пролонгацией договоров после завершения проектов. Это позволит выявить сотрудников с лучшими и худшими результатами, понять общую динамику удержания выручки по отделу, своевременно корректировать работу с клиентами после завершения проектов, а также принимать обоснованные управленческие решения — например, о премировании, дополнительном обучении или перераспределении клиентской базы между менеджерами.

## Описание данных

В данной работе будут использоваться два набора данных:

1. `prolongations.csv`
- `id` — id проекта;
- `month` — последний месяц реализации проекта;
- `AM` — ФИО ответственного аккаунт-менеджера;

2. `financial_data.csv`
- `id` - id проекта
- `Причина дубля` - причина, почему строки с одним и тем же id встречаются несколько раз;	
- `Колонки с названием месяца` -  сумма отгрузки проекта в данный месяц;
- `Account` - ФИО ответственного аккаунт-менеджера

## Содержимое проекта

1. **Шаг 1. Загрузка и первичное знакомство с данными**
   * Загрузить два набора данных: `prolongations.csv` и `financial_data.csv`
   * Изучить структуру таблиц, типы данных, количество записей
   * Выявить наличие дублей (особенно в `financial_data.csv` с учётом колонки «Причина дубля»)
   * Проверить соответствие списка проектов и ответственных менеджеров в двух источниках

<br>

2. **Шаг 2. Предобработка данных**
   * Привести названия месяцев к единому формату (тип «дата»)
   * Преобразовать значения отгрузки в числовой формат, обработав:
     * `«в ноль»` превращаем в 0 (но с особым правилом: для коэффициента берётся отгрузка предыдущего месяца, если все части оплаты равны 0)
     * `«стоп»` и `«end»` исключаем из расчётов пролонгаций
   * Соединить таблицы `financial_data.csv` и `prolongations.csv` 
   * Отфильтровать данные только за 2023 год

<br>

3. **Шаг 3. Расчёт коэффициентов пролонгации по месяцам**
   * Для каждого месяца (начиная с февраля 2023, так как нужен предыдущий месяц):
     * Определить проекты, завершившиеся в предыдущем месяце (`month` из `prolongations.csv`)
     * Найти среди них те, у которых есть отгрузка в текущем месяце (пролонгация в 1-й месяц)
     * Рассчитать коэффициент 1 - `сумма отгрузки за текущий месяц по таким проектам / сумма отгрузки за последний месяц реализации всех завершившихся проектов`
   * Для коэффициента 2 (пролонгация во второй месяц):
     * Взять проекты, завершившиеся 2 месяца назад
     * Отфильтровать те, у которых нет отгрузки через месяц после завершения
     * Среди них найти проекты с отгрузкой через 2 месяца
     * Рассчитать коэффициент 2 - `сумма отгрузки через 2 месяца / сумма отгрузки за последний месяц реализации проектов, не пролонгированных в первый месяц`
   * Повторить расчёты для каждого менеджера отдельно и для всего отдела

<br>

4. **Шаг 4. Агрегация результатов за год**
   * Рассчитать среднегодовые коэффициенты пролонгации (1 и 2) для каждого менеджера
   * Рассчитать те же коэффициенты для всего отдела в целом
   * Сравнить результаты менеджеров между собой и с общеотдельными показателями

<br>

5. **Шаг 5. Формирование аналитического отчёта**
   * Создать сводную таблицу по месяцам (строки — месяцы, столбцы — менеджеры + отдел, значения — коэффициент 1 и коэффициент 2)
   * Добавить итоговую таблицу за год с ранжированием менеджеров по коэффициентам
   * По желанию добавить дополнительные метрики:
     * количество завершившихся проектов на менеджера
     * количество пролонгаций (в штуках)
     * динамику коэффициентов по кварталам
   * Построить визуализации: линейные графики динамики коэффициентов, столбчатые диаграммы сравнения менеджеров
   * Экспортировать отчёт в Excel с форматированием 

<br>

6. **Шаг 6. Общие выводы**
   * Сформулировать краткие выводы для руководителя отдела (ключевые точки роста и лучшие практики)

## Этапы выполнения проекта
### Шаг 1. Загрузка и первичное знакомство с данными

Для начала импортируем библиотеки, которые нам будут необходимы для работы с данными:

- `pandas` - понадобится для загрузки датасетов и работы с ними
- `numpy` - понадобится для работы с значениями в датасетах
- `re` - для обработки  значений признаков

In [1]:
import pandas as pd
import numpy as np
import re

Загрузим датасеты и посмотрим их размеры, а также проверим какие признаки содержатся в датасетах

In [2]:
financial_data = pd.read_csv('financial_data.csv')
prolongations = pd.read_csv('prolongations.csv')

In [3]:
print(f"Размер prolongations: {prolongations.shape}")
print(f"Столбцы prolongations: {prolongations.columns}")
print(f"Размер financial_data: {financial_data.shape}")
print(f"Столбцы financial_data: {financial_data.columns}")

Размер prolongations: (477, 3)
Столбцы prolongations: Index(['id', 'month', 'AM'], dtype='object')
Размер financial_data: (451, 19)
Столбцы financial_data: Index(['id', 'Причина дубля', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2023',
       'Февраль 2023', 'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023',
       'Июль 2023', 'Август 2023', 'Сентябрь 2023', 'Октябрь 2023',
       'Ноябрь 2023', 'Декабрь 2023', 'Январь 2024', 'Февраль 2024',
       'Account'],
      dtype='object')


Данные успешно загружены, а признаки соответствуют описанию. Размер датасета `prolongations` - (477, 3), а размер датасета `financial_data` - (451, 19)

Далее изучим структуру таблиц и типы данных в них

In [4]:
print("\n1.2. Структура prolongations.csv:")
print(prolongations.info())
print("\nПервые 5 строк:")
print(prolongations.head())

print("\n1.2. Структура financial_data.csv:")
print(financial_data.info())
print("\nПервые 5 строк:")
print(financial_data.head())


1.2. Структура prolongations.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      477 non-null    int64 
 1   month   477 non-null    object
 2   AM      477 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.3+ KB
None

Первые 5 строк:
    id        month                             AM
0   42  ноябрь 2022   Васильев Артем Александрович
1  453  ноябрь 2022   Васильев Артем Александрович
2  548  ноябрь 2022      Михайлов Андрей Сергеевич
3   87  ноябрь 2022  Соколова Анастасия Викторовна
4  429  ноябрь 2022  Соколова Анастасия Викторовна

1.2. Структура financial_data.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 451 entries, 0 to 450
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             451 non-null    int64 
 1   Причина дубля  301 non-null

В `prolongations` у нас один признак имеет численный тип, а два признака - тип `object`. Все строки заполнены, пропусков по данным нет. В будущем можно попробовать уменьшить тип размерности признака `id` для экономия памяти.

В `financial_data` один признак имеет численный тип, остальные - тип `object`. В признаках `id` и `Account` нет пропусков, но в остальных признаках они присутствуют. Для удобства работы с признаками необходимо будет привести их названия в стиль **shake_case**. Также тип колонок с названиями месяцев надо будет преобразовать в тип datetime для удобства работы с данными. В будущем можно попробовать уменьшить тип размерности признака `id` для экономия памяти.

Изучим количество уникальных значений `id` в датасете `prolongations`

In [5]:
prolongations['id'].nunique()

313

Количество `id` проектов не совпадает с количеством строк датасета. Обратим на это внимание и в будущем сделаем очистку датасета, оставив одного актуального менеждера на проекте.

Проверим количество дублей в датасетах, а также посмотрим количество уникальных причин дубля по колонке `Причина дубля` в financial_data

In [6]:
dup_pro = prolongations['id'].duplicated().sum()
dup_fin = financial_data['id'].duplicated().sum()
print(f"Дубликаты id: prolongations={dup_pro}, financial_data={dup_fin}")

if 'Причина дубля' in financial_data.columns:
    reasons = financial_data['Причина дубля'].value_counts()
    print(f"Причины дублей: {reasons.to_dict()}")

Дубликаты id: prolongations=164, financial_data=137
Причины дублей: {'первая часть оплаты': 114, 'вторая часть оплаты': 99, 'доп работы': 38, 'основные работы': 38, 'изменение ЮЛ': 11, 'карты, банки': 1}


Можно заметить дубликаты в обоих таблицах, что считается нормальным, ведь в финансовых отчётах одна сделка часто разбита на несколько строк. В будущем впредобработке данных проведедём аггрегацию по `id` 

Дальше посмотрим совпадают ли списки проектов и менеджеров в двух источниках

In [7]:
projects_p = set(prolongations['id'].unique())
projects_f = set(financial_data['id'].unique())
print(f"Только в prolongations: {len(projects_p - projects_f)}")
print(f"Только в financial_data: {len(projects_f - projects_p)}")
print(f"Общих проектов: {len(projects_p & projects_f)}")

ams_p = set(prolongations['AM'].dropna().unique())
ams_f = set(financial_data.get('Account', pd.Series()).dropna().unique())
print(f"Менеджеры совпадают: {ams_p == ams_f}")

Только в prolongations: 0
Только в financial_data: 1
Общих проектов: 313
Менеджеры совпадают: True


По результатам проверки, общих проектов 313, и один из проектов находится только в `financial_data`, в будущем мы исключим этот проект и будем работать только с общими. Количество менеждеров в обоих таблицах совпадают.

Как итог были загружены датасеты, был проведём первичный осмотре данных, проверка на дубли и проверка совпадение списков проектов и менеждеров в двух источниках, намечены планы дальнейшей работы по предобработке данных.

### Шаг 2. Предобработка данных

Первым делом нам нужно привести даты в двух таблицах к единому формату: `YYYY-MM`. Так будет проще работать с данными в будущем. Посмотрим на уникальные значения `prolongations`

In [8]:
prolongations['month'].unique()

array(['ноябрь 2022', 'декабрь 2022', 'январь 2023', 'февраль 2023',
       'март 2023', 'апрель 2023', 'май 2023', 'июнь 2023', 'июль 2023',
       'август 2023', 'сентябрь 2023', 'октябрь 2023', 'ноябрь 2023',
       'декабрь 2023'], dtype=object)

Всего 14 уникальных значений и 12 уникальных месяцев. Создадим словарь с названием меяцев, каждому из которых будет присвоено число, затем напишем финкцию, которая будет переводить даты в нужный нам формат и применим её на `month` в таблице `prolongations`:

In [9]:
# Словарь для перевода месяцев
month_mapping = {
    'январь': 1, 'февраль': 2, 'март': 3, 'апрель': 4,
    'май': 5, 'июнь': 6, 'июль': 7, 'август': 8,
    'сентябрь': 9, 'октябрь': 10, 'ноябрь': 11, 'декабрь': 12
}

def convert_to_datetime(date_str):
    parts = date_str.split()
    month_name = parts[0].lower()
    year = int(parts[1])
    month_num = month_mapping[month_name]
    return pd.Timestamp(year=year, month=month_num, day=1)

# Применение к столбцу
prolongations['month_complited'] = prolongations['month'].apply(convert_to_datetime).dt.to_period('M')
prolongations.drop('month', axis=1, inplace=True)
prolongations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype    
---  ------           --------------  -----    
 0   id               477 non-null    int64    
 1   AM               477 non-null    object   
 2   month_complited  477 non-null    period[M]
dtypes: int64(1), object(1), period[M](1)
memory usage: 11.3+ KB


Теперь у нас появился новый столбец с типом `период`. Проверим корректность данных:

In [10]:
prolongations['month_complited'].unique()

<PeriodArray>
['2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05',
 '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12']
Length: 14, dtype: period[M]

Данные преобразованы корректно в необходимый формат. Также сделаем очистку менеджеров, чтобы оставить одного менеждера на проект. Сделаем сортировку по `month_complited`, чтобы оставить последнего менеждера проекта

In [11]:
prolongations = prolongations.sort_values('month_complited').drop_duplicates('id', keep='last')

Теперь каждому проекту присвоен один последний менеждер

Теперь надо преобразовать в такой же формат данные в `financial_data`. Задача усложняется тем, что в этом датасете месяца являются признаками. С помощью `melt` преобразуем признаки в строки и поместим их во временный признак `month_str`, значения этих признаков разместим в другом признаке `shippment`. Затем с помощью функции выше приведём данные в необходимый формат: 

In [12]:
month_columns = [month_col for month_col in financial_data.columns 
                 if month_col not in ['id','Причина дубля','Account']]
df_long = financial_data.melt(
    id_vars=['id', 'Причина дубля', 'Account'], 
    value_vars=month_columns,                     # Эти столбцы превратятся в строки
    var_name='month_str',                         # Новый столбец с именами старых столбцов
    value_name='shipment'                     # Новый столбец со значениями
)

In [13]:
df_long['year_month'] = df_long['month_str'].apply(convert_to_datetime).dt.to_period('M')
df_long.drop('month_str', axis=1, inplace=True)
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7216 entries, 0 to 7215
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype    
---  ------         --------------  -----    
 0   id             7216 non-null   int64    
 1   Причина дубля  4816 non-null   object   
 2   Account        7216 non-null   object   
 3   shipment       2596 non-null   object   
 4   year_month     7216 non-null   period[M]
dtypes: int64(1), object(3), period[M](1)
memory usage: 282.0+ KB


In [14]:
df_long['year_month'].unique()

<PeriodArray>
['2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05',
 '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12',
 '2024-01', '2024-02']
Length: 16, dtype: period[M]

Данные преобразованы корректно. Следующим шагом обработаем пропуски, а также изменим тип данных у признака `shipment`, и удалим из `shipment` проекты со значением `стоп` или `end`, сделаем корректную замеу данных со значением `в ноль`:

In [15]:
fin = df_long.copy()

fin['shipment'] = fin['shipment'].astype(str).str.strip().str.lower()

# помечаем проекты со stop/end
stop_ids = fin.loc[fin['shipment'].isin(['стоп', 'end']), 'id'].unique()

# удаляем все строки таких проектов
fin = fin[~fin['id'].isin(stop_ids)].copy()

def convert_number(s):
    if pd.isna(s):
        return 0
        
    s_clean = s.strip().lower()

    if s_clean == 'в ноль':
        return np.nan  # nan будет означать, что нужно взять значение из предыдущего месяца
        
    # Очистка от лишних символов
    cleaned = re.sub(r'[^\d,\-\.]', '', s_clean)
    cleaned = cleaned.replace(',', '.')
    try:
        return float(cleaned)
    except:
        return 0
        
fin['temp'] = fin['shipment'].apply(convert_number)

fin = fin.sort_values(['id', 'year_month'])

prev_id = None
prev_value = 0
new_shipments = []

for idx, row in fin.iterrows():
    current_id = row['id']
    
    # Если новый проект - сбрасываем prev_value
    if current_id != prev_id:
        prev_value = 0
    
    # Обрабатываем значение
    if pd.isna(row['temp']):  # nan означает 'в ноль'
        new_shipments.append(prev_value)  # берём значение из предыдущего месяца
    else:
        val = row['temp'] if isinstance(row['temp'], (int, float)) else 0
        new_shipments.append(val)
        prev_value = val  # запоминаем для следующего месяца
    
    prev_id = current_id

# Добавляем исправленные значения
fin['shipment'] = new_shipments
fin = fin.drop('temp', axis=1)

Данные скорректированы, пропуски отсутствуют. Сделаем группировку по `id` и `year_month`

In [16]:
fin_grouped = fin.groupby(
    ['id', 'year_month'], 
    as_index=False
)['shipment'].sum()

Соединим оба датасета, добавим новый признак `manager`, и удалим из объединённого датасета дубликаты по `id` и `year_month`, так как нем нужен один менеждер на один проект

In [17]:
df = fin_grouped.merge(prolongations[['id', 'AM', 'month_complited']], 
                       on='id', 
                       how='inner')

df = df.rename(columns={'AM': 'manager'})

Отфильтруем данные, оставив только до декабря 2023 года:

In [18]:
# оставляем конец 2022 и 2023
df = df[df['year_month'] <= '2023-12']

Создадим сводную таблицу, который будет готов к формулам пролонгации. Для создания индексов таблицы передадим `id`, `manager` и `month_complited`, в качестве колонки для группировки - `year_month`, c помощью `sum` вычислим сумму отгрузки по каждому месяцу для каждого проекта для каждого менеджера

In [19]:
final_table = df.pivot_table(
    index=['id', 'manager', 'month_complited'], 
    columns='year_month', 
    values='shipment', 
    aggfunc='sum' 
).reset_index()



In [20]:
final_table

year_month,id,manager,month_complited,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12
0,15,Иванова Мария Сергеевна,2023-06,439280.0,439280.0,102433.75,102433.75,102433.75,138158.0,138158.0,102433.75,0.0,0.0,0.0,0.0,0.0,0.00
1,16,Иванова Мария Сергеевна,2022-12,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00
2,31,Васильев Артем Александрович,2022-12,55100.0,55100.0,0.00,44775.00,44775.00,44775.0,44775.0,44775.00,44775.0,44775.0,44775.0,44775.0,44775.0,44775.00
3,39,Попова Екатерина Николаевна,2023-12,137700.0,137700.0,149206.50,149206.50,149206.50,149206.5,149206.5,149206.50,149206.5,149206.5,149206.5,149206.5,149206.5,149206.50
4,42,Васильев Артем Александрович,2022-11,36220.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,1001,Кузнецов Михаил Иванович,2023-11,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,290000.0,0.00
254,1004,без А/М,2023-12,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,34000.0,0.00
255,1006,Смирнова Ольга Владимировна,2023-12,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,145195.0,140495.0,140495.00
256,1012,Петрова Анна Дмитриевна,2023-11,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,98492.0,109442.52


В итоге были скорректированы форматы данных типа даты, была произведена предобработка `shipment`, удалены проекты с `end` или `стоп`, была создана сводная таблица для дальнейших расчётов.

### Шаг 3. Расчёт коэффициентов пролонгации по месяцам

Теперь займёмся расчётом необходимых коэффициентов: 

1. Сумму отгрузки проектов, завершившихся в апреле (за апрель) и сумму отгрузки тех проектов завершившихся в апреле, у которых есть отгрузка в мае (за май). Коэффициент – отношение второй суммы к первой.

2. Сумму проектов, завершившихся в марте, у которых нет отгрузки в апреле (за март) и сумму отгрузки тех проектов, завершившихся в марте, у которых нет отгрузки в апреле но есть в мае (за май). Коэффициент – отношение второй суммы к первой.

Коэффициенты необходимо посчитать как для каждого менеджера, так и для всего одтела в целом в рамках месяца и года.

Для начала займёмся расчётами первого коэффициента для всего отдела и каждого менеждера, а затем рассчётами второго коэффициента для всего отдела и каждого менеждера.

Подготовим отсортированный массив с месяцами из сводной таблицы

In [21]:
months = sorted([col for col in final_table.columns if isinstance(col, pd.Period)])

Займёмся расчётом первого коэффициента для всего отдела в целом. Сначала создаём датасет с проектами, которые завершились в прошлом месяце, затем также создаём отдельно датасет с данными о тех месяцах, у которых есть выручка в текущем месяце. Потом мы у этих двух датасетов считаем выручки (у превого датасета по прошлому месяцу, а у второго по текущему соответствено), и затем вычистываем коэффициент. Все результатысохраняем в датасет `rate1_df`

In [22]:
rate1 = []

for i in range(1, len(months)):
    curr = months[i]
    prev = months[i-1]

    # проекты, завершенные в prev
    ended = final_table[final_table['month_complited'] == prev]

    # пролонгация
    renewed = ended[ended[curr] > 0]

    sum_base = ended[prev].sum()
    sum_renew = renewed[curr].sum()

    k1 = sum_renew / sum_base if sum_base > 0 else 0

    rate1.append({
        'month': curr,
        'k1': k1
    })

rate1_df = pd.DataFrame(rate1)

Аналогично проводятся расчёты коэффициента, но уже для каждого менеджера. Результаты сохраянются в `rate1_mgr_df`

In [23]:
rate1_mgr = []

for i in range(1, len(months)):
    curr = months[i]
    prev = months[i-1]

    ended = final_table[final_table['month_complited'] == prev]

    for mgr in ended['manager'].unique():
        mgr_df = ended[ended['manager'] == mgr]

        renewed = mgr_df[mgr_df[curr] > 0]

        sum_base = mgr_df[prev].sum()
        sum_renew = renewed[curr].sum()

        k1 = sum_renew / sum_base if sum_base > 0 else 0

        rate1_mgr.append({
            'manager': mgr,
            'month': curr,
            'k1': k1
        })

rate1_mgr_df = pd.DataFrame(rate1_mgr)

Для рассчёта второго кэффициента потребуется значения за два месяца до актуального. Проекты не должны быть закончены за 2 месяца до актуального, не пролонгированы за месяц до актуального, но пролонгированы в текущем месяце. Коэффициент считается как сумма отгрузкизавершившихся проектов за 2 месяца до актуального в качестве знаменателя и сумма огрузки в актуальном месяце по проекта в качестве числителя. Результаты сохранены в `rate2_df`

In [24]:
rate2 = []

for i in range(2, len(months)):
    curr = months[i]
    prev1 = months[i-1]
    prev2 = months[i-2]

    ended = final_table[final_table['month_complited'] == prev2]

    # не пролонгированы в первый месяц
    not_renewed = ended[ended[prev1] == 0]

    # пролонгированы во второй
    renewed = not_renewed[not_renewed[curr] > 0]

    sum_base = not_renewed[prev2].sum()
    sum_renew = renewed[curr].sum()

    k2 = sum_renew / sum_base if sum_base > 0 else 0

    rate2.append({
        'month': curr,
        'k2': k2
    })

rate2_df = pd.DataFrame(rate2)

Аналогичны расчётся для каждого менеждера, добавляется только разбивка на менеждеров. Результат сохранён в `rate2_mgr_df`

In [25]:
rate2_mgr = []

for i in range(2, len(months)):
    curr = months[i]
    prev1 = months[i-1]
    prev2 = months[i-2]

    ended = final_table[final_table['month_complited'] == prev2]

    for mgr in ended['manager'].unique():
        mgr_df = ended[ended['manager'] == mgr]

        not_renewed = mgr_df[mgr_df[prev1] == 0]
        renewed = not_renewed[not_renewed[curr] > 0]

        sum_base = not_renewed[prev2].sum()
        sum_renew = renewed[curr].sum()

        k2 = sum_renew / sum_base if sum_base > 0 else 0

        rate2_mgr.append({
            'manager': mgr,
            'month': curr,
            'k2': k2
        })

rate2_mgr_df = pd.DataFrame(rate2_mgr)

In [26]:
rate2_mgr_df

,manager,month,k2
0,Васильев Артем Александрович,2023-01,0.000000
1,Михайлов Андрей Сергеевич,2023-01,0.000000
2,Попова Екатерина Николаевна,2023-01,0.000000
3,Соколова Анастасия Викторовна,2023-01,0.000000
4,Иванова Мария Сергеевна,2023-02,0.000000
...,...,...,...
56,Соколова Анастасия Викторовна,2023-11,0.000000
57,Соколова Анастасия Викторовна,2023-12,0.000000
58,Смирнова Ольга Владимировна,2023-12,0.000000
59,Васильев Артем Александрович,2023-12,0.484859


Итого расчитаны и сохранены результаты двух коэффицентов по месяцам для каждого менеждера и всего отдела в целом.

### Шаг 4. Агрегация результатов за год

Следующим шагом необходимо посчитать эти же коэффициенты, но с агрегацией результатов за год.
Формула немного видоизменится, но суть примерно та же. Вот пример расчёта первого коэффициента для всего отдела и каждого менеджера: 

$$k1\_year =
(сумма\_всех\_пролонгаций\_за\_год)
/
(сумма\_всех\_завершённых\_проектов\_за\_год)$$


$$k1\_manager\_year =
(сумма\_всех\_пролонгаций\_менеджера)
/
(сумма\_всех\_завершённых\_проектов\_менеджера)$$

In [27]:
total_base = 0
total_renew = 0

for i in range(1, len(months)):
    curr = months[i]
    prev = months[i-1]

    ended = final_table[final_table['month_complited'] == prev]
    renewed = ended[ended[curr] > 0]

    total_base += ended[prev].sum()
    total_renew += renewed[curr].sum()

k1_year = total_renew / total_base

In [28]:
k1_mgr_year = []

for mgr in final_table['manager'].unique():
    total_base = 0
    total_renew = 0

    for i in range(1, len(months)):
        curr = months[i]
        prev = months[i-1]

        ended = final_table[
            (final_table['month_complited'] == prev) &
            (final_table['manager'] == mgr)
        ]

        renewed = ended[ended[curr] > 0]

        total_base += ended[prev].sum()
        total_renew += renewed[curr].sum()

    k1 = total_renew / total_base if total_base > 0 else 0

    k1_mgr_year.append({'manager': mgr, 'k1_year': k1})

k1_mgr_year_df = pd.DataFrame(k1_mgr_year)

In [29]:
total_base = 0
total_renew = 0

for i in range(2, len(months)):
    curr = months[i]
    prev1 = months[i-1]
    prev2 = months[i-2]

    ended = final_table[final_table['month_complited'] == prev2]
    not_renewed = ended[ended[prev1] == 0]
    renewed = not_renewed[not_renewed[curr] > 0]

    total_base += not_renewed[prev2].sum()
    total_renew += renewed[curr].sum()

k2_year = total_renew / total_base

In [30]:
k2_mgr_year = []

for mgr in final_table['manager'].unique():
    total_base = 0
    total_renew = 0

    for i in range(2, len(months)):
        curr = months[i]
        prev1 = months[i-1]
        prev2 = months[i-2]

        ended = final_table[
            (final_table['month_complited'] == prev2) &
            (final_table['manager'] == mgr)
        ]

        not_renewed = ended[ended[prev1] == 0]
        renewed = not_renewed[not_renewed[curr] > 0]

        total_base += not_renewed[prev2].sum()
        total_renew += renewed[curr].sum()

    k2 = total_renew / total_base if total_base > 0 else 0

    k2_mgr_year.append({'manager': mgr, 'k2_year': k2})

k2_mgr_year_df = pd.DataFrame(k2_mgr_year)

Итого рассчитаны и сохранены коэффициенты пролонгации за год для каждо менеждера и отдела.

### Шаг 5. Формирование аналитического отчёта

Коэффициенты посчитаны, и теперь осталось сформировать аналитический отчёт

Создадим сводные таблицы для результатам по месяцам для каждого менеждера

In [31]:
monthly_k1 = rate1_mgr_df.copy()
monthly_k2 = rate2_mgr_df.copy()

pivot_k1 = monthly_k1.pivot_table(
    index='month',
    columns='manager',
    values='k1'
)

pivot_k2 = monthly_k2.pivot_table(
    index='month',
    columns='manager',
    values='k2'
)

Объединим данные для всего отдела в целом

In [32]:
monthly_dep = rate2_df.merge(
    rate2_df,
    on='month'
)

Объединим данные за год

In [33]:
year_total = k1_mgr_year_df.merge(
    k2_mgr_year_df,
    on='manager'
)

А также добавим ранжирование чтобы можно было, не всматриваясь в цифры, определить, у какого из менеджера коэффициент выше

In [34]:
year_total['rank_k1'] = year_total['k1_year'].rank(ascending=False, method='min')
year_total['rank_k2'] = year_total['k2_year'].rank(ascending=False, method='min')

In [35]:
year_total

,manager,k1_year,k2_year,rank_k1,rank_k2
0,Иванова Мария Сергеевна,0.139951,0.000000,7.0,5.0
1,Васильев Артем Александрович,0.041469,0.031265,8.0,2.0
2,Попова Екатерина Николаевна,0.199581,0.013606,6.0,4.0
3,Смирнова Ольга Владимировна,0.985915,0.185375,2.0,1.0
4,Кузнецов Михаил Иванович,0.538092,0.000000,3.0,5.0
5,Михайлов Андрей Сергеевич,0.413383,0.000000,4.0,5.0
6,Соколова Анастасия Викторовна,0.292048,0.020263,5.0,3.0
7,Федорова Марина Васильевна,0.000000,0.000000,9.0,5.0
8,без А/М,0.000000,0.000000,9.0,5.0
9,Петрова Анна Дмитриевна,1.111182,0.000000,1.0,5.0


Также создадим датафрейм для результатов коэффициентов в рамках года для всего отдела

In [36]:
dept_summary = pd.DataFrame({
    'k1_year': [k1_year],
    'k2_year': [k2_year]
})

Все данные готовы к экспорту, займёмся этим

In [37]:
with pd.ExcelWriter('report.xlsx') as writer:
    pivot_k1.to_excel(writer, sheet_name='K1 по месяцам')
    pivot_k2.to_excel(writer, sheet_name='K2 по месяцам')
    monthly_dep.to_excel(writer, sheet_name='Итог отдела по месяцам')
    year_total.to_excel(writer, sheet_name='Итоги по менеджерам')
    dept_summary.to_excel(writer, sheet_name='Итог отдела по году')

Результаты работы собраны в единый excel-документ

### Шаг 6. Общие выводы и рекомендации

Лидером по K1 является Петрова Анна Дмитриевна (K1 = 1,11), демонстрирующая рост объёмов после пролонгации. Второй результат у Смирновой О.В. (K1 = 0,986), которая также лидирует по K2 (0,185), эффективно возвращая клиентов через второй месяц.

Средние результаты показывают Кузнецов М.И. и Михайлов А.С. (K1 ~0,41–0,54), при этом K2 у них отсутствует. Соколова А.В. и Попова Е.Н. имеют низкий K1 (0,29 и 0,20), но частичный K2, что указывает на работу с отложенными пролонгациями. Наиболее слабые показатели у Ивановой М.С. (0,14) и Васильева А.А. (0,04), при этом у последнего наблюдается небольшой K2, что говорит о смещении работы в сторону поздних возвратов. Федорова М.В. и нераспределённые проекты пролонгаций не показали.

Основные точки роста: развитие практик ранней пролонгации (K1), тиражирование подходов лидеров, усиление работы с менеджерами с низким K1, а также развитие сценариев возврата клиентов во второй месяц, так как K2 остаётся низким и представляет потенциальный резерв выручки. В целом отдел демонстрирует стабильную динамику, а уточнение методологии расчётов позволило получить более объективную картину для управленческих решений.